<a href="https://colab.research.google.com/github/kondreddygarivani-bit/project-8/blob/main/W8sample_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install torch
!pip install langchain-text-splitters
!pip install langchain-community
!pip install langchain-huggingface

In [3]:
# IMPORT LIBRARIES
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# READ DATASET
with open("data.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("\nDATASET TEXT\n")
print(text)

# SPLIT TEXT INTO CHUNKS
splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

docs = splitter.split_text(text)

print("\nTEXT CHUNKS\n")
print(docs)


DATASET TEXT

 

TEXT CHUNKS

[]


In [7]:
%%writefile data.txt
This is a sample document about Retrieval-Augmented Generation (RAG).
RAG models combine the strengths of retrieval-based and generative models.
They first retrieve relevant documents and then generate a response based on these documents.
This approach helps in generating more factual and grounded answers, reducing hallucinations.
RAG is particularly useful for tasks like question answering and content generation where accuracy is paramount.

Overwriting data.txt


In [37]:
# SPLIT TEXT INTO CHUNKS
splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

documents = splitter.split_text(text)

# PRINT CHUNKS
print("\nTEXT CHUNKS\n")
print(docs)



TEXT CHUNKS

[]


In [44]:
# IMPORT LIBRARIES
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# SAMPLE TEXT
text = """
Artificial Intelligence is simulation of human intelligence.

Machine Learning is subset of AI.

Deep Learning uses neural networks.

RAG stands for Retrieval Augmented Generation.

NLP helps computers understand human language.
"""

# CREATE DOCUMENT OBJECTS
documents = [Document(page_content=text)]

print("\nDOCUMENTS CREATED\n")
print(documents)

# SPLIT DOCUMENTS
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

docs = splitter.split_documents(documents)

print("\nCHUNKS\n")

for doc in docs:
    print(doc.page_content)

# LOAD EMBEDDING MODEL
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("\nEMBEDDING MODEL LOADED\n")

# TEST EMBEDDING
test_embedding = embedding_model.embed_query("hello")

print("\nTEST EMBEDDING LENGTH\n")
print(len(test_embedding))

# CREATE FAISS VECTOR STORE
vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

print("\nVECTOR DATABASE CREATED SUCCESSFULLY\n")


DOCUMENTS CREATED

[Document(metadata={}, page_content='\nArtificial Intelligence is simulation of human intelligence.\n\nMachine Learning is subset of AI.\n\nDeep Learning uses neural networks.\n\nRAG stands for Retrieval Augmented Generation.\n\nNLP helps computers understand human language.\n')]

CHUNKS

Artificial Intelligence is simulation of human intelligence.

Machine Learning is subset of AI.
Deep Learning uses neural networks.

RAG stands for Retrieval Augmented Generation.
NLP helps computers understand human language.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



EMBEDDING MODEL LOADED


TEST EMBEDDING LENGTH

384

VECTOR DATABASE CREATED SUCCESSFULLY



In [60]:
# CREATE CHUNKS
documents = splitter.create_documents([text])

# PRINT CHUNKS
print("\nDOCUMENT CHUNKS\n")

for doc in docs:
    print(doc.page_content)

# LOAD EMBEDDING MODEL
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("\nEMBEDDING MODEL LOADED\n")

# CREATE VECTOR DATABASE
vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

print("\nVECTOR DATABASE CREATED\n")

# CREATE RETRIEVER
retriever = vectorstore.as_retriever()

# LOAD LLM
qa_pipeline = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

print("\nRAG MODEL READY\n")

# USER QUESTION
query = input("Enter your question: ")

# RETRIEVE DOCUMENTS
retrieved_docs = retriever.invoke(query)

print("\nRETRIEVED DOCUMENTS\n")

context = ""

for doc in retrieved_docs:
    print(doc.page_content)
    context += doc.page_content + "\n"

# PROMPT
prompt = f"""
Context:
{context}

Question:
{query}

Answer the question based only on the context.
"""

# GENERATE ANSWER
response = qa_pipeline(
    prompt,
    max_new_tokens=50
)

# FINAL ANSWER
print("\nFINAL ANSWER\n")
print(response[0]['generated_text'])


DOCUMENT CHUNKS

Artificial Intelligence is simulation of human intelligence.

Machine Learning is subset of AI.
Deep Learning uses neural networks.

RAG stands for Retrieval Augmented Generation.
NLP helps computers understand human language.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



EMBEDDING MODEL LOADED


VECTOR DATABASE CREATED



Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl


RAG MODEL READY

Enter your question: what is AI?

RETRIEVED DOCUMENTS

Artificial Intelligence is simulation of human intelligence.

Machine Learning is subset of AI.
NLP helps computers understand human language.
Deep Learning uses neural networks.

RAG stands for Retrieval Augmented Generation.


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL ANSWER


Context:
Artificial Intelligence is simulation of human intelligence.

Machine Learning is subset of AI.
NLP helps computers understand human language.
Deep Learning uses neural networks.

RAG stands for Retrieval Augmented Generation.


Question:
what is AI?

Answer the question based only on the context.

